# YOLO11n ONNX export and test fixtures

Run this notebook top to bottom on Google Colab (Runtime > Run all). A CPU runtime is enough. It produces `yolo_assets.zip`, which the last cell downloads.

Extract the zip straight into the repo root. It contains the folders `models/` and `tests/fixtures/`, so each file lands here:

| File | Destination |
| --- | --- |
| `yolo11n-320.onnx` | `models/yolo11n-320.onnx` |
| `yolo11n-640.onnx` | `models/yolo11n-640.onnx` |
| `bus.jpg` | `tests/fixtures/bus.jpg` |
| `bus-3s.mp4` | `tests/fixtures/bus-3s.mp4` (3 s, 30 fps, with audio) |
| `bus-3s-noaudio.mp4` | `tests/fixtures/bus-3s-noaudio.mp4` (no audio track) |
| `bus-3s-rot90.mp4` | `tests/fixtures/bus-3s-rot90.mp4` (rotation metadata 90) |

Every verification cell uses `assert`, so a failure stops the run with a clear error. Do not download the zip if any cell fails.

Licence note: YOLO11n is AGPL-3.0 (Ultralytics). See `models/README.md`.

In [ ]:
!pip install -q ultralytics==8.3.40 onnx

In [ ]:
import os
import shutil
from ultralytics import YOLO

OUT = '/content/out'
MODELS = f'{OUT}/models'
FIX = f'{OUT}/tests/fixtures'
shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(MODELS, exist_ok=True)
os.makedirs(FIX, exist_ok=True)

for imgsz in (320, 640):
    model = YOLO('yolo11n.pt')
    # Ultralytics writes yolo11n.onnx next to yolo11n.pt and overwrites it on every export,
    # so move it to its final name before the next export runs.
    exported = model.export(format='onnx', imgsz=imgsz, opset=17, simplify=True, dynamic=False, nms=False)
    dest = f'{MODELS}/yolo11n-{imgsz}.onnx'
    shutil.move(str(exported), dest)
    print(imgsz, '->', dest, round(os.path.getsize(dest) / 1e6, 1), 'MB')

In [ ]:
import onnx

EXPECTED_OUTPUT = {320: [1, 84, 2100], 640: [1, 84, 8400]}

def dims(value_info):
    return [d.dim_value for d in value_info.type.tensor_type.shape.dim]

for imgsz, expected in EXPECTED_OUTPUT.items():
    path = f'{MODELS}/yolo11n-{imgsz}.onnx'
    assert os.path.exists(path), f'missing {path}'
    m = onnx.load(path)
    onnx.checker.check_model(m)
    assert len(m.graph.input) == 1, f'{path}: expected 1 input, got {len(m.graph.input)}'
    assert len(m.graph.output) == 1, f'{path}: expected 1 output, got {len(m.graph.output)}'
    inp, out = dims(m.graph.input[0]), dims(m.graph.output[0])
    size_mb = os.path.getsize(path) / 1e6
    print(f'yolo11n-{imgsz}.onnx: input {inp}, output {out}, {size_mb:.1f} MB')
    assert inp == [1, 3, imgsz, imgsz], f'{path}: input shape {inp}'
    assert out == expected, f'{path}: output shape {out}, expected {expected}'
    assert 9 <= size_mb <= 12, f'{path}: size {size_mb:.1f} MB outside 9-12 MB'
print('model shapes OK')

In [ ]:
!curl -fsSL -o {FIX}/bus.jpg https://ultralytics.com/images/bus.jpg

from PIL import Image
img = Image.open(f'{FIX}/bus.jpg')
print('bus.jpg', img.size, round(os.path.getsize(f'{FIX}/bus.jpg') / 1024), 'KB')
assert img.size[0] > 0 and img.size[1] > 0
assert img.size[0] % 2 == 0 and img.size[1] % 2 == 0, 'libx264 yuv420p needs even dimensions'

In [ ]:
import re
import subprocess

# 3 s, 30 fps, H.264 plus a 440 Hz AAC tone.
!ffmpeg -y -loglevel error -loop 1 -i {FIX}/bus.jpg -f lavfi -i sine=frequency=440:sample_rate=44100 -t 3 -r 30 -c:v libx264 -pix_fmt yuv420p -g 30 -c:a aac -b:a 96k -shortest {FIX}/bus-3s.mp4

# Same video, no audio track.
!ffmpeg -y -loglevel error -loop 1 -i {FIX}/bus.jpg -t 3 -r 30 -c:v libx264 -pix_fmt yuv420p -g 30 -an {FIX}/bus-3s-noaudio.mp4

# Same streams, copied, with rotation metadata 90.
!ffmpeg -y -loglevel error -i {FIX}/bus-3s.mp4 -c copy -metadata:s:v:0 rotate=90 {FIX}/bus-3s-rot90.mp4

def rotation_info(path):
    r = subprocess.run(
        ['ffprobe', '-v', 'error', '-select_streams', 'v:0',
         '-show_entries', 'stream_side_data=rotation:stream_tags=rotate',
         '-of', 'default=nw=1', path],
        capture_output=True, text=True, check=True)
    return r.stdout.strip()

ROT = f'{FIX}/bus-3s-rot90.mp4'
if not re.search(r'(rotation|rotate)=-?90', rotation_info(ROT)):
    # Newer ffmpeg builds can ignore the rotate tag. Write a display matrix instead.
    print('rotate tag not honoured, retrying with -display_rotation 90')
    subprocess.run(
        ['ffmpeg', '-y', '-loglevel', 'error', '-display_rotation:v:0', '90',
         '-i', f'{FIX}/bus-3s.mp4', '-c', 'copy', ROT],
        check=True)
print('clips written')

In [ ]:
def probe(path, *args):
    r = subprocess.run(['ffprobe', '-v', 'error', *args, path], capture_output=True, text=True, check=True)
    return r.stdout.strip()

def frame_count(path):
    return probe(path, '-count_frames', '-select_streams', 'v:0',
                 '-show_entries', 'stream=nb_read_frames', '-of', 'csv=p=0')

def audio_streams(path):
    return probe(path, '-select_streams', 'a', '-show_entries', 'stream=codec_name', '-of', 'csv=p=0')

WANT_AUDIO = {'bus-3s.mp4': True, 'bus-3s-noaudio.mp4': False, 'bus-3s-rot90.mp4': True}

for name, want_audio in WANT_AUDIO.items():
    path = f'{FIX}/{name}'
    assert os.path.exists(path), f'missing {path}'
    frames = frame_count(path)
    audio = audio_streams(path)
    print(f'{name}: frames={frames}, audio={audio or "none"}')
    assert frames == '90', f'{name}: expected 90 frames, got {frames!r}'
    assert bool(audio) == want_audio, f'{name}: audio presence is {bool(audio)}, expected {want_audio}'

rot = rotation_info(ROT)
print('bus-3s-rot90.mp4 rotation side data / tags:')
print(rot or '  (none found)')
assert re.search(r'(rotation|rotate)=-?90', rot), 'bus-3s-rot90.mp4 has no 90 degree rotation metadata'
print('clips OK')

In [ ]:
import zipfile
from google.colab import files

ZIP = '/content/yolo_assets.zip'
if os.path.exists(ZIP):
    os.remove(ZIP)
shutil.make_archive(ZIP[:-4], 'zip', OUT)  # root_dir=OUT, so the zip holds models/ and tests/fixtures/

with zipfile.ZipFile(ZIP) as z:
    names = sorted(n for n in z.namelist() if not n.endswith('/'))
for n in names:
    print(n)
assert names == sorted([
    'models/yolo11n-320.onnx',
    'models/yolo11n-640.onnx',
    'tests/fixtures/bus.jpg',
    'tests/fixtures/bus-3s.mp4',
    'tests/fixtures/bus-3s-noaudio.mp4',
    'tests/fixtures/bus-3s-rot90.mp4',
]), f'unexpected zip contents: {names}'

files.download(ZIP)